In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)


In [0]:
df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/demo/default/delta_practice1")

In [0]:
df = spark.read.format("delta").load("/Volumes/demo/default/delta_practice1")
df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
new_df=spark.createDataFrame([(5, "C005", "Camera", 30000)],columns)
new_df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



In [0]:
new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/demo/default/delta_practice1")

In [0]:
spark.read.format("delta")\
    .load("/Volumes/demo/default/delta_practice1").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/demo/default/delta_practice1")

delta_table.update(
    condition = "id = 2",
    set = {"amount": "18000"}
)
delta_table.update(
    condition = "id = 2",
    set = {"product": "'Camera'"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
spark.read.format("delta")\
    .load("/Volumes/demo/default/delta_practice1").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  2|       C002| Camera| 18000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



In [0]:
delta_table.delete("id=1")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.read.format("delta")\
    .load("/Volumes/demo/default/delta_practice1").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  2|       C002| Camera| 18000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



In [0]:
target_df = spark.read.format("delta").load("/Volumes/demo/default/delta_practice1")
target_df.display()

id,customer_id,product,amount
2,C002,Camera,18000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


In [0]:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]
updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
delta_table.alias("target").merge(
    updates_df.alias("updates"),
    "target.id = updates.id")\
    .whenMatchedUpdate(set={
        "amount": "updates.amount"
    }).whenNotMatchedInsert(values={
        "id": "updates.id",
        "customer_id": "updates.customer_id",
        "product": "updates.product",
        "amount": "updates.amount"
    }).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
new_data=spark.createDataFrame([(6,"C007","Tablet",12000)],columns)
new_data.show()


+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  6|       C007| Tablet| 12000|
+---+-----------+-------+------+



In [0]:
new_data.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/demo/default/delta_practice1")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/demo/default/delta_practice1")
df_read.display()

id,customer_id,product,amount
2,C002,Camera,18000
4,C004,Laptop,55000
5,C005,Camera,30000
3,C003,Tablet,22000
6,C006,Watch,8000
6,C007,Tablet,12000


In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
26,2026-04-14T15:53:02.000Z,77971296220999,skshajith007@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1922876678890877),5ad1aba1-69fe-41a0-a4a3-d965761b28ab,0414-150608-koskv59p-v2n,25,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1240)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
25,2026-04-14T15:47:41.000Z,77971296220999,skshajith007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1922876678890877),1e534e69-1aa2-487a-904b-f124aab20d49,0414-150608-koskv59p-v2n,24,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3826, p25FileSize -> 1374, numDeletionVectorsRemoved -> 1, minFileSize -> 1374, numAddedFiles -> 1, maxFileSize -> 1374, p75FileSize -> 1374, p50FileSize -> 1374, numAddedBytes -> 1374)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
24,2026-04-14T15:47:38.000Z,77971296220999,skshajith007@gmail.com,MERGE,"Map(predicate -> [""(id#15689L = id#15693L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1922876678890877),1e534e69-1aa2-487a-904b-f124aab20d49,0414-150608-koskv59p-v2n,23,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4190, materializeSourceTimeMs -> 396, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1673, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1990)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
23,2026-04-14T15:45:52.000Z,77971296220999,skshajith007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1922876678890877),280d95a2-5aa7-453c-babd-a5b1c7d9ce87,0414-150608-koskv59p-v2n,22,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2591, p25FileSize -> 1351, numDeletionVectorsRemoved -> 1, minFileSize -> 1351, numAddedFiles -> 1, maxFileSize -> 1351, p75FileSize -> 1351, p50FileSize -> 1351, numAddedBytes -> 1351)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
22,2026-04-14T15:45:50.000Z,77971296220999,skshajith007@gmail.com,UPDATE,"Map(predicate -> [""(id#15212L = 2)""])",null,List(1922876678890877),280d95a2-5aa7-453c-babd-a5b1c7d9ce87,0414-150608-koskv59p-v2n,20,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1240, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2315, conflictDetectionTimeMs -> 172, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1414, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1240, rewriteTimeMs -> 900)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
21,2026-04-14T15:45:49.000Z,77971296220999,skshajith007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1922876678890877),2310c910-6c84-44d9-a381-7bb0e2c81b8b,0414-150608-koskv59p-v2n,20,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2591, p25FileSize -> 1351, numDeletionVectorsRemoved -> 1, minFileSize -> 1351, numAddedFiles -> 1, maxFileSize -> 1351, p75FileSize -> 1351, p50FileSize -> 1351, numAddedBytes -> 1351)",null,Dat

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/demo/default/delta_practice1")
df_old.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]